# **MINI BOOTCAMP DATA ENGINEERING PROJECT PART 1**

---


*DIGITAL SKILL FAIR 44.0: FACULTY OF DATA ENGINEERING*

*   **Project Name: End-to-end Job Market Trends Pipeline**
*   **by Fauzan Dharmawan**

---



**⚠️ Disclaimer**

This project is created **solely for educational and learning purposes** .

All data collected and analyzed in this project are **limited to publicly accessible information** and comply with each website’s robots.txt policy.

The dataset used in this project only covers the **period of up to 30 days ago, and no personal, confidential, or sensitive user data** are collected, stored, or shared.

This project does **not represent**, nor is it **affiliated** with LinkedIn in any form.

Check robot.txt:
https://www.linkedin.com/robots.txt



---



## **Requests Job Portal url link**

*200 (OK)*: The request succeeded and the page content was returned successfully.

*403 (Forbidden)*: The server understood the request but denied access — often due to permission limits or anti-bot protection.

*404 (Not Found)*: The requested URL doesn’t exist or the page is unavailable.

### *Linkedin*

⚙️ Note: Make sure you have applied the “Past Month” (Last 30 Days) filter on the LinkedIn Jobs search page before copy the url to ensure the results align with the intended timeframe.

In [2]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
import random
import time

job_title = ""
location = "Indonesia"
geoId = "102478259"

# Scrape all job IDs across all pages
linkedinJobs_idList = []
start = 0
step = 25
max_pages = 1000  # batas maksimal halaman

while True:
    linkedin_url = f"https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?keywords={job_title}&location={location}&geoId={geoId}&f_TPR=r2592000&position=1&pageNum=0&start={start}"
    response = requests.get(linkedin_url)
    print(f"Scraping start={start} | Status: {response.status_code}")

    if response.status_code != 200 or not response.text.strip():
        print("No more data or error fetching page.")
        break

    soup = bs(response.content, "html.parser")
    linkedinPage_jobs = soup.find_all("li")
    if not linkedinPage_jobs:
        print("No more job listings found.")
        break

    for linkedinJobs in linkedinPage_jobs:
        base_card_div = linkedinJobs.find("div", {"class": "base-card"})
        if base_card_div and base_card_div.get("data-entity-urn"):
            linkedinJobs_id = base_card_div.get("data-entity-urn").split(":")[3]
            linkedinJobs_idList.append(linkedinJobs_id)

    start += step
    if start >= max_pages * step:
        print(f"Reached page limit ({max_pages} pages). Stopping scraping.")
        break

    time.sleep(1)

print(f"Total job IDs collected: {len(linkedinJobs_idList)}")

# Scrape each job’s detail
linkedinJob_list = []
for linkedinJob_id in linkedinJobs_idList:
    linkedin_job_url = f"https://id.linkedin.com/jobs-guest/jobs/api/jobPosting/{linkedinJob_id}"
    linkedin_job_response = requests.get(linkedin_job_url)

    print(f"Scraping Job ID {linkedinJob_id} | Status: {linkedin_job_response.status_code}")

    # delay scraping time to avoid rate limit
    time.sleep(random.uniform(3, 7))

    if linkedin_job_response.status_code != 200:
        continue

    linkedin_job_soup = bs(linkedin_job_response.content, "html.parser")

    linkedinJob_post = {"linkedinJob_id": linkedinJob_id}

    # job title
    linkedinJob_post["linkedinJob_title"] = (
        linkedin_job_soup.find("h2", {"class": "top-card-layout__title"}).text.strip()
        if linkedin_job_soup.find("h2", {"class": "top-card-layout__title"})
        else None
    )

    # company name
    linkedinJob_post["linkedinCompany_name"] = (
        linkedin_job_soup.find("a", {"class": "topcard__org-name-link topcard__flavor--black-link"}).text.strip()
        if linkedin_job_soup.find("a", {"class": "topcard__org-name-link topcard__flavor--black-link"})
        else None
    )

    # location
    linkedinJob_post["linkedinJob_location"] = (
        linkedin_job_soup.find("span", {"class": "topcard__flavor topcard__flavor--bullet"}).text.strip()
        if linkedin_job_soup.find("span", {"class": "topcard__flavor topcard__flavor--bullet"})
        else None
    )

    # job criteria
    for item in linkedin_job_soup.find_all("li", {"class": "description__job-criteria-item"}):
        header = item.find("h3", {"class": "description__job-criteria-subheader"})
        value = item.find("span", {"class": "description__job-criteria-text description__job-criteria-text--criteria"})

        if header and value:
            header_text = header.get_text(strip=True)
            value_text = value.get_text(strip=True)

            if "Seniority" in header_text or "Tingkat" in header_text:
                linkedinJob_post["linkedinJob_seniority"] = value_text
            elif "Employment" in header_text or "Jenis" in header_text:
                linkedinJob_post["linkedinJob_employmentType"] = value_text
            elif "Function" in header_text or "Fungsi" in header_text:
                linkedinJob_post["linkedinJob_function"] = value_text
            elif "Industr" in header_text:
                linkedinJob_post["linkedinJob_industries"] = value_text

    # number of applicant
    linkedinJob_post["linkedinNumber_applicants"] = (
        linkedin_job_soup.find ("span", {"class" : "num-applicants__caption"}).text.strip()
        if linkedin_job_soup.find ("span", {"class" : "num-applicants__caption"})
        else None
    )


    # posted date
    linkedinJob_post["linkedinJob_postedTime"] = (
        linkedin_job_soup.find("span", {"class": "posted-time-ago__text"}).text.strip()
        if linkedin_job_soup.find("span", {"class": "posted-time-ago__text"})
        else None
    )

    linkedinJob_list.append(linkedinJob_post)

# Convert to DataFrame
linkedinJob_df = pd.DataFrame(linkedinJob_list)
print(linkedinJob_df.head())

Scraping start=0 | Status: 200
Scraping start=25 | Status: 200
Scraping start=50 | Status: 200
Scraping start=75 | Status: 200
Scraping start=100 | Status: 200
Scraping start=125 | Status: 200
Scraping start=150 | Status: 200
Scraping start=175 | Status: 200
Scraping start=200 | Status: 200
Scraping start=225 | Status: 200
Scraping start=250 | Status: 200
Scraping start=275 | Status: 200
Scraping start=300 | Status: 200
Scraping start=325 | Status: 200
Scraping start=350 | Status: 200
Scraping start=375 | Status: 200
Scraping start=400 | Status: 200
Scraping start=425 | Status: 200
Scraping start=450 | Status: 200
Scraping start=475 | Status: 200
Scraping start=500 | Status: 200
Scraping start=525 | Status: 200
Scraping start=550 | Status: 200
Scraping start=575 | Status: 200
Scraping start=600 | Status: 200
Scraping start=625 | Status: 200
Scraping start=650 | Status: 200
Scraping start=675 | Status: 200
Scraping start=700 | Status: 200
Scraping start=725 | Status: 200
Scraping start=

In [3]:
linkedinJob_df

,linkedinJob_id,linkedinJob_title,linkedinCompany_name,linkedinJob_location,linkedinJob_seniority,linkedinJob_employmentType,linkedinJob_function,linkedinJob_industries,linkedinNumber_applicants,linkedinJob_postedTime
0,4375724428,Finance Accounting Staff,Kalbe Nutritionals (PT Sanghiang Perkasa),Karawang Barat,Tingkat pemula,Penuh waktu,"Akuntansi/Audit, Keuangan",Jasa Penyedia Makanan dan Minuman,None,4 hari yang lalu
1,4370258070,Officer Development Program (ODP) Retail Banking,PT. BANK NEGARA INDONESIA (Persero) Tbk.,"Jakarta Raya, Indonesia",Tingkat pemula,Penuh waktu,"Analis, Manajemen, Keuangan","Perbankan, Jasa Keuangan",None,2 minggu yang lalu
2,4376705498,Staff GA (Pulogadung),PT. Sarimelati Kencana Tbk.,Jakarta Timur,Tingkat pemula,Kontrak,"Sumber Daya Manusia, Administratif, Manajemen","Manufaktur, Manufaktur Makanan dan Minuman, Ad...",None,2 hari yang lalu
3,4375151664,Import Staff,PT. Surya TOTO Indonesia Tbk,Area DKI Jakarta,Tidak Berlaku,Penuh waktu,"Manajemen, Manufaktur",Konstruksi,189 pelamar,2 minggu yang lalu
4,4368981113,Future Bankers Development Program,PT Bank Sinarmas Tbk,"Jakarta Raya, Indonesia",Tingkat pemula,Penuh waktu,"Pemasaran, Penjualan, Hubungan Masyarakat",Perbankan,None,3 minggu yang lalu
...,...,...,...,...,...,...,...,...,...,...
393,4375905946,Project Officer Event,Waste4Change,Kota Bekasi,Tidak Berlaku,Penuh waktu,"Manajemen, Manufaktur",Jasa Terkait Lingkungan,43 pelamar,16 jam yang lalu
394,4370794940,Store Drafter,Blibli,Area DKI Jakarta,Tingkat pemula,Kontrak,"Desain, Seni/Kreatif, Teknologi Informasi","Teknologi, Informasi, dan Internet",190 pelamar,2 minggu yang lalu
395,4369331863,Legal Officer,Taman Safari Indonesia,Jakarta Selatan,Asosiasi,Penuh waktu,"Legal, Administratif, Konsultasi","Pariwisata dan Perhotelan, Pengaturan Perjalan...",None,3 minggu yang lalu
396,4366853397,Accountant,Halliburton,Bangko,Senior tingkat menengah,Penuh waktu,"Akuntansi/Audit, Keuangan",Minyak dan Gas,None,4 minggu yang lalu


## **EXPORT INTO EXCEL**

In [5]:
linkedinJob_df.to_excel("linkedin_jobs.xlsx", index=False)

After generating the xlsx file from the scraping process, download it and upload into Microsoft SQL Server as a staging table. The staging layer is used to store raw structured data before further transformation using Apache Spark.

*Microsoft SQL Server acts as an intermediate storage layer between the raw data source and the transformation process.*